In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

from scipy.stats import zscore
from sklearn.cluster import KMeans

from scipy.io import mmread
from scipy.sparse import csr_matrix, hstack,vstack
from scipy.stats import chi2_contingency

import glob

from statsmodels.stats.multitest import multipletests

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/hicluster'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
bins_path = '/u/project/cluo/terencew/reference/hg38_igvf/bed/windows/25kb/GRCh38.25kb.bed'
bins = pd.read_csv(bins_path, sep='\t', header=None)
bins.index = [f'{x}_{y}' for x,y in zip(bins[0], bins[1])]
bins.shape

In [ ]:
bins.head()

In [ ]:
indir = f'{projdir}/csv/domains/batch'

batch = '000'

boundary = mmread(f'{indir}/{batch}.boundary.mtx')
boundary

In [ ]:
n_batches = 274
batches = [str(i).zfill(3) for i in range(n_batches)]

In [ ]:
def load_topdom(s):
    boundary = mmread(f'{indir}/{s}.boundary.mtx')
    return boundary

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(load_topdom, batches), total=len(batches)))

In [ ]:
merged_boundary = csr_matrix(hstack(results))
merged_boundary

### make metadata with clusters

In [ ]:
tmp_cells = []
for s in batches:
    tmp_cells.append(pd.read_csv(f'{projdir}/txt/batch/domains/{s}.txt', header=None, index_col=0))
    
cells = pd.concat(tmp_cells, axis=0).index
cells = [x.split('/')[-1].split('.')[0] for x in cells]
len(cells)

In [ ]:
meta = pd.DataFrame(index=cells)
meta['donor'] = [x.split('-')[1][1:4] for x in meta.index]
meta['donor'].replace({'39D' : 'C39'}, inplace=True)

meta['time'] = [x.split('-')[1][4:] for x in meta.index]
meta['time'].replace({'5' : 'D5'}, inplace=True)

meta.shape

In [ ]:
# indir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/mc/'
# clusters = pd.read_csv(f'{indir}/csv/label_transfer/xgboost_time_v7.csv', sep='\t', index_col=0)
# clusters['cluster_donor'] = [f'{x}_{y}' for x,y in zip(clusters['cluster'], clusters['line'])]
# clusters.shape

In [ ]:
ips_clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/label_transfer/snm3c_ips_subtype.csv'
ips_clusters = pd.read_csv(ips_clusters_path, sep='\t', index_col=0)
ips_clusters = ips_clusters.rename(columns={'ips_subtype' : 'ips_cluster'})
ips_clusters['ips_cluster_donor'] = [f'{x}_{y}' for x,y in zip(ips_clusters['ips_cluster'], ips_clusters['line'])]
ips_clusters.shape

In [ ]:
final_meta = meta.reindex(ips_clusters.index)
final_meta['ips_cluster'] = ips_clusters['ips_cluster']
final_meta['ips_cluster_donor'] = ips_clusters['ips_cluster_donor']
final_meta.shape

In [ ]:
final_meta.head()

In [ ]:
tmp_clusters = ips_clusters.reindex(meta.index)
mask = ~tmp_clusters['time'].isna()
tmp_clusters = tmp_clusters[mask]
tmp_clusters.shape

In [ ]:
final_boundary = merged_boundary[:,mask.values]
n_boundaries = np.ravel(final_boundary.sum(axis=0))
tmp_clusters['n_boundaries'] = n_boundaries

### compute some quick boundary stats

In [ ]:
tmp_clusters

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'ips_cluster'
y_lab = 'n_boundaries'
hue = 'line'

ax = sns.boxplot(data=tmp_clusters, x=x_lab, y=y_lab, hue=hue, showfliers=False)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('', fontsize=16)
ax.set_title('', fontsize=20)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

### look into chi-squared test

In [ ]:
tmp_clusters.shape, final_boundary.shape

In [ ]:
tmp_clusts = tmp_clusters['ips_cluster'].unique()
boundary_df = pd.DataFrame(index=bins.index)
non_boundary_df = pd.DataFrame(index=bins.index)

for s in tmp_clusts:
    indices = np.where(tmp_clusters['ips_cluster'] == s)[0]
    cell_boundaries = np.ravel(final_boundary[:,indices].sum(axis=1))
    boundary_df[s] = cell_boundaries
    non_boundary_df[s] = indices.shape[0] - cell_boundaries

In [ ]:
boundary_df.shape, non_boundary_df.shape

In [ ]:
mask = boundary_df.sum(axis=1) > 0
final_boundary_df = boundary_df.loc[mask]
final_non_boundary_df = non_boundary_df.loc[mask]
mask.sum()

In [ ]:
boundary_sum = final_boundary_df.values
non_boundary_sum = final_non_boundary_df.values

In [ ]:
i = 0
tmp_a = boundary_sum[i]
tmp_b = non_boundary_sum[i]

table = np.array([tmp_a, tmp_b]).T

chi2, pval, dof, expected = chi2_contingency(table, correction=False)

In [ ]:
tmp_indices = np.arange(boundary_sum.shape[0])

def run_chi2(i):
    tmp_a = boundary_sum[i]
    tmp_b = non_boundary_sum[i]
    table = np.array([tmp_a, tmp_b]).T
    chi2, pval, dof, expected = chi2_contingency(table)
    return chi2, pval

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(run_chi2, tmp_indices), total=len(tmp_indices)))

In [ ]:
chi2 = [x[0] for x in results]
pvals = np.array([x[1] for x in results])
pvals_adj = multipletests(pvals, alpha=0.05, method='fdr_bh')[1]

In [ ]:
cutoff = 0.01
(pvals < cutoff).sum(), (pvals_adj < cutoff).sum()

In [ ]:
tmp_indices = (pvals_adj < cutoff)
tmp_indices.sum()

In [ ]:
final_boundary_df[tmp_indices].head()

In [ ]:
final_non_boundary_df[tmp_indices].head()

### write out

In [ ]:
sig_hits = pd.concat([final_boundary_df[tmp_indices], final_non_boundary_df[tmp_indices]], axis=1)
sig_hits.columns = ['IPS_main_boundary', 'IPS_alt_boundary',
                    'IPS_main_domain', 'IPS_alt_domain']
sig_hits.shape

In [ ]:
sig_hits.head()

In [ ]:
sig_hits.to_csv(f'{projdir}/csv/tads/ips_cluster_sig.csv', sep='\t')

In [ ]:
boundary_sum = sig_hits['IPS_main_boundary'] + sig_hits['IPS_alt_boundary']
boundary_sum.mean()

In [ ]:
sns.histplot(boundary_sum)

In [ ]:
boundary_sum

In [ ]:
!date